## Solver definitions

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, List

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation


@dataclass
class Config:
    """Configuration for the Minimal 2D State-Resolution Simulation."""

    grid_h: int = 64
    grid_w: int = 64
    steps: int = 300
    w: int = 6  # rolling history window length; W >= 3 recommended

    alpha: float = 10.0
    rho_c: float = 0.5

    u0: float = 1e-3
    seed_sign: int = 1  # +1 or -1

    eps_rho: float = 1e-12
    eps_c: float = 1e-12
    eps_m: float = 1e-12
    eps_x: float = 1e-12

    store_u_frames: bool = True
    store_R_frames: bool = True
    capture_every: int = 1

    # Model motion-law weights for the reduced scalar update
    # Proposed scalar closure:
    # - delta responds to local relational contrast
    # - flux redistributes recursive influence via div(Phi * grad(R_signed))
    # - decoherence damps carried state where Gamma_D_proxy is large
    # - decoherence weight strongly shifts regimes
    lambda_delta: float = 1.00        # (1.00)
    lambda_flux: float = 0.50         # (0.50)
    lambda_decoherence: float = 0.25  # (0.25)

    # Discrete operator mode for the transport branch:
    # - "matched" keeps the current Sobel-like isotropic gradient/divergence pair
    # - "naive" uses simple axis-aligned central differences with the same
    #   zero-padded open-boundary policy
    operator_mode: str = "matched"

    summary_every: int = 10

    # Visualization / export
    fps: int = 12
    interval_ms: int = 80
    output_mp4: str = "2D_u_field.mp4"
    plot_title: str = "2D U-Field"
    cmap: str = "coolwarm"

    # Display-only color scaling for MP4 export.
    # Uses one global symmetric percentile across all stored frames so that
    # low-to-mid magnitude structure is more visible without changing the
    # simulation values themselves.
    export_display_percentile: float = 99.5

    # Display-only axis tick controls for exported MP4.
    show_axis_ticks: bool = True

    def validate(self) -> None:
        if self.grid_h <= 0 or self.grid_w <= 0:
            raise ValueError("grid_h and grid_w must be positive integers.")
        if self.steps < 0:
            raise ValueError("steps must be non-negative.")
        if self.w < 3:
            raise ValueError("w must be >= 3.")
        if self.alpha <= 0:
            raise ValueError("alpha must be > 0.")
        if self.u0 <= 0:
            raise ValueError("u0 must be > 0.")
        if self.seed_sign not in (-1, 1):
            raise ValueError("seed_sign must be either +1 or -1.")
        if self.capture_every <= 0:
            raise ValueError("capture_every must be >= 1.")
        if self.summary_every <= 0:
            raise ValueError("summary_every must be >= 1.")
        if self.fps <= 0:
            raise ValueError("fps must be >= 1.")
        if self.interval_ms <= 0:
            raise ValueError("interval_ms must be >= 1.")
        if not (0.0 < self.export_display_percentile <= 100.0):
            raise ValueError("export_display_percentile must be in (0, 100].")
        if self.lambda_delta < 0:
            raise ValueError("lambda_delta must be >= 0.")
        if self.lambda_flux < 0:
            raise ValueError("lambda_flux must be >= 0.")
        if self.lambda_decoherence < 0:
            raise ValueError("lambda_decoherence must be >= 0.")
        if self.operator_mode not in ("matched", "naive"):
            raise ValueError('operator_mode must be either "matched" or "naive".')


class MinimalSim:
    """
    Minimal 2D State-Resolution Simulation

    Notes:
    - "step" means resolved state succession, not ontological clock time

    Present design:
    - local relational contrast is reduced to delta = neighbor_mean(u) - u
    - rho is a scalar curvature-density surrogate built from local contrast
    - Phi(rho) is the main activation response
    - chi is a smooth odd orientation carrier built from the current scalar state
    - F_flip is diagnostics-only and does not influence the motion law
    - the motion update is a lean scalar combination of local delta response,
      recursive flux redistribution, and decoherence loss
    """

    def __init__(self, config: Config):
        self.cfg = config
        self.cfg.validate()

        self.h = self.cfg.grid_h
        self.wd = self.cfg.grid_w
        self.W = self.cfg.w

        # -------------------------
        # State arrays
        # -------------------------
        self.u_curr = np.zeros((self.h, self.wd), dtype=np.float64)


        # -------------------------
        # Core observables
        # -------------------------
        self.u_mean = np.zeros_like(self.u_curr)
        self.delta = np.zeros_like(self.u_curr)
        self.rho = np.zeros_like(self.u_curr)
        self.phi = np.zeros_like(self.u_curr)

        # History-derived diagnostics
        # - C_rec, M_mag, and A_hist contribute only through Gamma_proxy
        # - F_flip is reporting-only and does not influence Gamma_proxy,
        #   R_proxy, Gamma_D_proxy, or the motion update
        self.C_rec = np.zeros_like(self.u_curr)
        self.F_flip = np.zeros_like(self.u_curr)
        self.M_mag = np.zeros_like(self.u_curr)
        self.A_hist = np.zeros_like(self.u_curr)

        # Reduced scalar state summaries used in the update
        # Gamma_proxy   -> local activity-density summary (Gamma)
        # R_proxy       -> recursive/history state summary (R)
        # Gamma_D_proxy -> damping/decoherence summary (Gamma_D)
        self.Gamma_proxy = np.zeros_like(self.u_curr)
        self.R_proxy = np.zeros_like(self.u_curr)
        self.Gamma_D_proxy = np.zeros_like(self.u_curr)

        # Smooth local orientation carrier and signed recursive-depth proxy.
        # chi is an odd carrier built from the current scalar field.
        # R_signed = chi * R_proxy is used only for the flux branch so that
        # recursive redistribution flips under global sign reversal.
        self.chi = np.zeros_like(self.u_curr)
        self.R_signed = np.zeros_like(self.u_curr)

        # Gradient / influence observables
        self.grad_phi_x = np.zeros_like(self.u_curr)
        self.grad_phi_y = np.zeros_like(self.u_curr)
        self.grad_R_x = np.zeros_like(self.u_curr)
        self.grad_R_y = np.zeros_like(self.u_curr)

        # 2D proxy for recursive influence.
        # This code stores the x/y components of a reduced influence field
        # derived from Phi(rho) and the gradient of R_proxy.
        self.Ix = np.zeros_like(self.u_curr)
        self.Iy = np.zeros_like(self.u_curr)

        # Scalar projection used by the update law
        # flux_proj is the scalar redistribution term built from the
        # divergence of the reduced recursive influence field:
        #     flux_proj = div(I),  with  I = Phi * grad(R_signed)
        self.flux_proj = np.zeros_like(self.u_curr)
        self.decoherence_term = np.zeros_like(self.u_curr)

        # Optional reporting magnitudes
        self.grad_phi_mag = np.zeros_like(self.u_curr)
        self.grad_R_mag = np.zeros_like(self.u_curr)
        self.I_mag = np.zeros_like(self.u_curr)



        # -------------------------
        # History buffer
        # history_buffer[0] = most recent
        # -------------------------
        self.history_buffer = np.zeros((self.W, self.h, self.wd), dtype=np.float64)

        # -------------------------
        # Optional frame storage
        # -------------------------
        self.u_frames: List[np.ndarray] = []
        self.R_frames: List[np.ndarray] = []

        # -------------------------
        # Summary storage
        # -------------------------
        self.summaries: List[Dict[str, Any]] = []

        # -------------------------
        # Seed coordinates
        # -------------------------
        self.seed_i = self.h // 2
        self.seed_j = self.wd // 2

        # -------------------------
        # Precompute valid isotropic neighbor weights
        # -------------------------
        # Matched 3x3 Moore-stencil family:
        # cardinal neighbors carry weight 2, diagonal neighbors weight 1.
        # This supplies the normalization for the isotropic local mean used
        # to build both delta and rho.
        self._iso_neighbor_weight_sum = self._isotropic_neighbor_weight_sum(
            np.ones_like(self.u_curr)
        )

    # =========================================================
    # Helper functions
    # =========================================================
    @staticmethod
    def sign(x: np.ndarray | float) -> np.ndarray | int:
        """SIGN(x): +1 if x>0, -1 if x<0, 0 if x==0"""
        if isinstance(x, np.ndarray):
            return np.sign(x).astype(np.int8)
        return 1 if x > 0 else (-1 if x < 0 else 0)

    @staticmethod
    def clamp(x: np.ndarray | float, lo: float, hi: float) -> np.ndarray | float:
        return np.clip(x, lo, hi)

    @staticmethod
    def masked_mean(field: np.ndarray, mask: np.ndarray) -> float:
        """
        Mean of field over mask. Returns 0.0 if the mask is empty.
        """
        if np.any(mask):
            return float(np.mean(field[mask]))
        return 0.0

    def logistic_activation(self, rho_value: np.ndarray) -> np.ndarray:
        """
        logistic activation response:
            Phi(rho) = 1 / (1 + exp(-alpha * (rho - rho_c)))
        """
        x = self.cfg.alpha * (rho_value - self.cfg.rho_c)
        x = self.clamp(x, -60.0, 60.0)  # numerical safety only
        return 1.0 / (1.0 + np.exp(-x))

    def _neighbor_sum(self, field: np.ndarray) -> np.ndarray:
        """
        Sum over valid in-domain Moore neighbors (8-neighbor sum).
        No wraparound. No special edge forcing.
        """
        p = np.pad(field, 1, mode="constant", constant_values=0.0)
        s = (
            p[:-2, :-2] + p[:-2, 1:-1] + p[:-2, 2:] +
            p[1:-1, :-2]               + p[1:-1, 2:] +
            p[2:, :-2]  + p[2:, 1:-1]  + p[2:, 2:]
        )
        return s

    def _isotropic_neighbor_weight_sum(self, field: np.ndarray) -> np.ndarray:
        """
        Weighted Moore-neighbor sum using the same 3x3 isotropic family that
        underlies the matched gradient/divergence operators.

        Weights:
            cardinal neighbors -> 2
            diagonal neighbors -> 1

        Center cell is excluded.
        Open boundaries are preserved through zero padding.
        """
        N, S, E, W, NE, NW, SE, SW = self._stencil_views(field)

        weighted_sum = (
            2.0 * (N + S + E + W)
            + (NE + NW + SE + SW)
        )
        return weighted_sum


    def _stencil_views(
        self,
        field: np.ndarray,
    ) -> tuple[
        np.ndarray, np.ndarray, np.ndarray, np.ndarray,
        np.ndarray, np.ndarray, np.ndarray, np.ndarray,
    ]:
        """
        Return the 8 Moore-neighborhood directional views of a field using
        open boundaries (constant zero padding).

        Output order:
            N, S, E, W, NE, NW, SE, SW
        """
        p = np.pad(field, 1, mode="constant", constant_values=0.0)

        N = p[:-2, 1:-1]
        S = p[2:, 1:-1]
        E = p[1:-1, 2:]
        W = p[1:-1, :-2]

        NE = p[:-2, 2:]
        NW = p[:-2, :-2]
        SE = p[2:, 2:]
        SW = p[2:, :-2]

        return N, S, E, W, NE, NW, SE, SW

    def neighbor_mean(self, field: np.ndarray) -> np.ndarray:
        """
        Matched isotropic local mean built from the same 3x3 Moore-stencil
        family used by the gradient/divergence operators.

        NOTE:
        neighbor_mean() now means "matched isotropic local mean" rather than
        a uniform 8-neighbor average. The name is retained to minimize churn in
        the rest of the model code.

        This replaces the previous uniform Moore-neighbor average with a
        weighted local mean:
            cardinal neighbors -> 2
            diagonal neighbors -> 1

        The center cell is excluded. Open boundaries are preserved.
        """
        weighted_sum = self._isotropic_neighbor_weight_sum(field)
        return np.divide(
            weighted_sum,
            self._iso_neighbor_weight_sum,
            out=np.zeros_like(field),
            where=self._iso_neighbor_weight_sum > 0,
        )


    def compute_boundary_mask(self, field: np.ndarray) -> np.ndarray:
        """
        Boolean mask for signed cells that touch at least one opposite-sign
        Moore neighbor.

        Zero-valued cells are not counted as boundary cells.
        """
        s = self.sign(field)
        N, S, E, W, NE, NW, SE, SW = self._stencil_views(s)

        signed_mask = s != 0
        opposite_touch = (
            (N * s < 0) |
            (S * s < 0) |
            (E * s < 0) |
            (W * s < 0) |
            (NE * s < 0) |
            (NW * s < 0) |
            (SE * s < 0) |
            (SW * s < 0)
        )

        return signed_mask & opposite_touch


    def compute_recurrence_similarity(self) -> np.ndarray:
        a = self.history_buffer[:-1]
        b = self.history_buffer[1:]

        numerator = np.sum(a * b, axis=0)
        sum_a2 = np.sum(a * a, axis=0)
        sum_b2 = np.sum(b * b, axis=0)

        denom = np.sqrt(sum_a2) * np.sqrt(sum_b2) + self.cfg.eps_c
        value = numerator / denom
        return self.clamp(value, -1.0, 1.0)

    def compute_sign_flip_regularity(self) -> np.ndarray:
        a = self.history_buffer[:-1]
        b = self.history_buffer[1:]

        sa = self.sign(a)
        sb = self.sign(b)

        active_pairs = (np.abs(a) + np.abs(b)) > 0.0
        signed_pairs = (sa != 0) & (sb != 0) & active_pairs
        flip_pairs = (sa != sb) & signed_pairs

        valid_pairs = np.sum(signed_pairs, axis=0)
        flip_count = np.sum(flip_pairs, axis=0)

        result = np.zeros_like(self.u_curr)
        np.divide(
            flip_count,
            valid_pairs,
            out=result,
            where=valid_pairs > 0,
        )
        return result

    def compute_magnitude_consistency(self) -> np.ndarray:
        mags = np.abs(self.history_buffer)
        mean_mag = np.mean(mags, axis=0)
        std_mag = np.std(mags, axis=0)

        value = 1.0 - std_mag / (mean_mag + self.cfg.eps_m)
        return self.clamp(value, 0.0, 1.0)

    def compute_activity_gate(self) -> np.ndarray:
        """
        Persistence-sensitive activity diagnostic.

        mean_mag / (max_mag + eps_m)

        High when motion is carried across the window.
        Low when motion appears only as brief spikes.
        """
        mags = np.abs(self.history_buffer)
        max_mag = np.max(mags, axis=0)
        mean_mag = np.mean(mags, axis=0)

        value = mean_mag / (max_mag + self.cfg.eps_m)
        value[max_mag == 0.0] = 0.0
        return self.clamp(value, 0.0, 1.0)

    def compute_gamma_proxy(self) -> np.ndarray:
        """
        - Gamma rises with persistent carried recurrence
        - Gamma is reduced when carry is brief or magnitude is inconsistent
        - constructed from C_rec, M_mag, and A_hist only
        - F_flip is reporting-only and does not enter Gamma
        """
        value = self.A_hist * self.M_mag * np.abs(self.C_rec)
        return self.clamp(value, 0.0, 1.0)

    def compute_r_proxy(self) -> np.ndarray:
        """
        R_proxy = Phi(rho) * Gamma_proxy
        """
        value = self.phi * self.Gamma_proxy
        return self.clamp(value, 0.0, 1.0)

    def compute_orientation_carrier(self, field: np.ndarray) -> np.ndarray:
        """
        Smooth odd local orientation carrier for the scalar model.

        Model reduction:
        - chi carries local sign/orientation information without introducing a
          hard discontinuity at zero.
        - chi is bounded in [-1, 1].
        - the local scale uses the same isotropic local mean already used by
          delta and rho.

        This carrier is used only to sign R_proxy for the
        flux branch:
            R_signed = chi * R_proxy
        """
        denom = self.cfg.eps_x + np.abs(field) + np.abs(self.u_mean)
        value = np.divide(
            field,
            denom,
            out=np.zeros_like(field),
            where=denom > 0,
        )
        return self.clamp(value, -1.0, 1.0)

    def compute_scalar_gradient_matched(self, field: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        """
        Return (d/dx, d/dy) using the matched isotropic 3x3 Moore-stencil
        approximation with open boundaries.

        The stencil is Sobel-like:
            d/dx ~ [[-1, 0, 1],
                    [-2, 0, 2],
                    [-1, 0, 1]] / 8

            d/dy ~ [[-1, -2, -1],
                    [ 0,  0,  0],
                    [ 1,  2,  1]] / 8
        """
        N, S, E, W, NE, NW, SE, SW = self._stencil_views(field)

        grad_x = (
            2.0 * (E - W)
            + (NE + SE - NW - SW)
        ) / 8.0

        grad_y = (
            2.0 * (S - N)
            + (SE + SW - NE - NW)
        ) / 8.0

        return grad_x, grad_y

    def compute_scalar_gradient_naive(self, field: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        """
        Return (d/dx, d/dy) using a simpler axis-aligned central-difference
        stencil with the same zero-padded open-boundary policy.

        This intentionally drops the diagonal coupling used by the matched
        operator pair so that A/B tests isolate the value of the isotropic
        transport discretization.
        """
        N, S, E, W, NE, NW, SE, SW = self._stencil_views(field)

        grad_x = (E - W) / 2.0
        grad_y = (S - N) / 2.0

        return grad_x, grad_y

    def compute_scalar_gradient(self, field: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        """
        Dispatch the transport-branch gradient operator selected by cfg.operator_mode.
        """
        if self.cfg.operator_mode == "naive":
            return self.compute_scalar_gradient_naive(field)
        return self.compute_scalar_gradient_matched(field)

    def compute_divergence_matched(self, vx: np.ndarray, vy: np.ndarray) -> np.ndarray:
        """
        Scalar projection of a 2D vector field back to the proxy-model grid using
        the same matched isotropic 3x3 Moore-stencil family as
        compute_scalar_gradient_matched().

        This keeps gradient and divergence in one discrete geometry, reducing
        directional bias from mixed operator families.
        """
        Nvx, Svx, Evx, Wvx, NEvx, NWvx, SEvx, SWvx = self._stencil_views(vx)
        Nvy, Svy, Evy, Wvy, NEvy, NWvy, SEvy, SWvy = self._stencil_views(vy)

        dvx_dx = (
            2.0 * (Evx - Wvx)
            + (NEvx + SEvx - NWvx - SWvx)
        ) / 8.0

        dvy_dy = (
            2.0 * (Svy - Nvy)
            + (SEvy + SWvy - NEvy - NWvy)
        ) / 8.0

        return dvx_dx + dvy_dy

    def compute_divergence_naive(self, vx: np.ndarray, vy: np.ndarray) -> np.ndarray:
        """
        Scalar projection of a 2D vector field using the same open-boundary
        policy as the matched version, but with a simpler axis-aligned
        central-difference stencil.
        """
        Nvx, Svx, Evx, Wvx, NEvx, NWvx, SEvx, SWvx = self._stencil_views(vx)
        Nvy, Svy, Evy, Wvy, NEvy, NWvy, SEvy, SWvy = self._stencil_views(vy)

        dvx_dx = (Evx - Wvx) / 2.0
        dvy_dy = (Svy - Nvy) / 2.0

        return dvx_dx + dvy_dy

    def compute_divergence(self, vx: np.ndarray, vy: np.ndarray) -> np.ndarray:
        """
        Dispatch the transport-branch divergence operator selected by cfg.operator_mode.
        """
        if self.cfg.operator_mode == "naive":
            return self.compute_divergence_naive(vx, vy)
        return self.compute_divergence_matched(vx, vy)

    def compute_gamma_d_proxy(self) -> np.ndarray:

        grad_phi_mag = np.sqrt(self.grad_phi_x**2 + self.grad_phi_y**2)
        scale = grad_phi_mag + np.abs(self.R_proxy)
        value = grad_phi_mag / (self.cfg.eps_x + scale)
        return self.clamp(value, 0.0, 1.0)

    # =========================================================
    # Observable pass for a given current state
    # =========================================================
    def compute_observables_for_current_state(self, field: np.ndarray) -> None:

        # Local relational geometry
        # u_mean is a matched isotropic local mean built from the same 3x3
        # stencil family used by the model gradient/divergence operators.
        # delta is the signed local contrast relative to that isotropic mean.
        self.u_mean = self.neighbor_mean(field)
        self.delta = self.u_mean - field

        # rho is a reduced local contrast measure, not a full geometric
        # curvature object. It is built from the same matched isotropic local
        # geometry as delta and is used only to drive the proxy-model activation
        # map Phi(rho).
        denom = self.cfg.eps_rho + np.abs(field) + np.abs(self.u_mean)
        self.rho = np.abs(self.delta) / denom
        self.phi = self.logistic_activation(self.rho)

        # History-derived observables
        # - C_rec, M_mag, and A_hist still contribute to the Gamma proxy
        # - F_flip is computed for diagnostics / reporting only
        # - F_flip does not enter Gamma_proxy, R_proxy, Gamma_D_proxy, or the motion update
        self.C_rec = self.compute_recurrence_similarity()
        self.F_flip = self.compute_sign_flip_regularity()
        self.M_mag = self.compute_magnitude_consistency()
        self.A_hist = self.compute_activity_gate()

        # model recursion proxies
        self.Gamma_proxy = self.compute_gamma_proxy()
        self.R_proxy = self.compute_r_proxy()

        # Smooth local orientation carrier for sign-neutral recursive transport.
        # R_proxy remains unsigned for reporting.
        # R_signed is used only in the flux branch so that recursive influence
        # flips under global sign reversal.
        self.chi = self.compute_orientation_carrier(field)
        self.R_signed = self.chi * self.R_proxy

        # Local gradient structure of the current proxy-model observables.
        # grad(R_signed) is used for the recursive influence branch so that the
        # resulting flux term is odd under global sign reversal.
        self.grad_phi_x, self.grad_phi_y = self.compute_scalar_gradient(self.phi)
        self.grad_R_x, self.grad_R_y = self.compute_scalar_gradient(self.R_signed)

        self.grad_phi_mag = np.sqrt(self.grad_phi_x**2 + self.grad_phi_y**2)
        self.grad_R_mag = np.sqrt(self.grad_R_x**2 + self.grad_R_y**2)

        # Recursive influence proxy 𝓘^μ = Φ ∇^μ 𝓡
        # Reduced 2D influence field built from Phi(rho) and grad(R_signed).
        # R_signed is used only for transport orientation;
        # R_proxy remains the reported scalar depth proxy.
        self.Ix = self.phi * self.grad_R_x
        self.Iy = self.phi * self.grad_R_y
        self.I_mag = np.sqrt(self.Ix**2 + self.Iy**2)

        # damping/decoherence summary Γ_D
        self.Gamma_D_proxy = self.compute_gamma_d_proxy()

    # =========================================================
    # Initialization
    # =========================================================
    def initialize(self) -> None:
        self.u_curr.fill(0.0)

        self.flux_proj.fill(0.0)
        self.decoherence_term.fill(0.0)

        self.u_curr[self.seed_i, self.seed_j] = float(self.cfg.seed_sign) * self.cfg.u0

        self.history_buffer[0] = self.u_curr.copy()
        self.history_buffer[1].fill(0.0)
        for k in range(2, self.W):
            self.history_buffer[k].fill(0.0)

        self.compute_observables_for_current_state(self.u_curr)

        self._capture_frames(step_index=0)
        self._capture_summary(step_index=0)

    # =========================================================
    # Per-step computation order
    # =========================================================
    def step(self, step_index: int) -> None:
        # Scalar model redistribution term built from the current
        # reduced recursive influence field:
        #     R_signed = chi * R_proxy
        #     I = Phi * grad(R_signed)
        #     flux_proj = div(I)
        #
        # chi is a smooth odd orientation carrier built from the current
        # scalar field. This keeps the flux branch sign-neutral under global
        # sign reversal without introducing a hard sign discontinuity.
        #
        # grad(.) and div(.) are both evaluated with the same matched
        # isotropic 3x3 Moore-stencil family to reduce lattice-axis bias.
        self.flux_proj = self.compute_divergence(self.Ix, self.Iy)

        # Lean scalar motion law:
        # - delta_term responds to local neighbor contrast
        # - flux_term redistributes recursive influence through div(I)
        # - decoherence_term damps carried state where Gamma_D_proxy is large
        delta_term = self.cfg.lambda_delta * self.phi * self.delta
        flux_term = self.cfg.lambda_flux * self.flux_proj
        self.decoherence_term = self.cfg.lambda_decoherence * self.Gamma_D_proxy * self.u_curr

        u_next = (
            self.u_curr
            + delta_term
            + flux_term
            - self.decoherence_term
        )

        self.u_curr = u_next.copy()

        self.history_buffer[1:] = self.history_buffer[:-1]
        self.history_buffer[0] = self.u_curr.copy()

        self.compute_observables_for_current_state(self.u_curr)

        self._capture_frames(step_index=step_index)
        self._capture_summary(step_index=step_index)

    # =========================================================
    # Main loop
    # =========================================================
    def run(self) -> Dict[str, Any]:
        self.initialize()

        for step_index in range(1, self.cfg.steps + 1):
            self.step(step_index)

        return {
            "u_curr": self.u_curr.copy(),
            "phi": self.phi.copy(),
            "rho": self.rho.copy(),
            "Gamma_proxy": self.Gamma_proxy.copy(),
            "R_proxy": self.R_proxy.copy(),
            "R_signed": self.R_signed.copy(),
            "chi": self.chi.copy(),
            "Gamma_D_proxy": self.Gamma_D_proxy.copy(),
            "Ix": self.Ix.copy(),
            "Iy": self.Iy.copy(),
            "summaries": self.summaries,
            "u_frames": self.u_frames,
            "R_frames": self.R_frames,
        }

    # =========================================================
    # Summaries / frame capture
    # =========================================================
    def compute_summary(self, step_index: int) -> Dict[str, Any]:
        boundary_mask = self.compute_boundary_mask(self.u_curr)
        signed_mask = self.sign(self.u_curr) != 0
        off_boundary_signed_mask = signed_mask & (~boundary_mask)

        boundary_occupancy_ratio = float(np.mean(boundary_mask))

        mean_rho_boundary = self.masked_mean(self.rho, boundary_mask)
        mean_rho_off_boundary = self.masked_mean(self.rho, off_boundary_signed_mask)
        rho_boundary_enrichment = mean_rho_boundary / (mean_rho_off_boundary + self.cfg.eps_rho)

        mean_F_flip_boundary = self.masked_mean(self.F_flip, boundary_mask)
        mean_F_flip_off_boundary = self.masked_mean(self.F_flip, off_boundary_signed_mask)
        bulk_oscillation_ratio = mean_F_flip_off_boundary / (mean_F_flip_boundary + self.cfg.eps_x)

        rho_boundary_share = float(
            np.sum(self.rho[boundary_mask]) /
            (np.sum(self.rho[signed_mask]) + self.cfg.eps_rho)
        )

        return {
            "step": step_index,

            "mean_abs_u": float(np.mean(np.abs(self.u_curr))),
            "mean_rho": float(np.mean(self.rho)),
            "mean_phi": float(np.mean(self.phi)),

            "mean_Gamma": float(np.mean(self.Gamma_proxy)),
            "mean_R": float(np.mean(self.R_proxy)),
            "mean_Gamma_D": float(np.mean(self.Gamma_D_proxy)),

            "mean_grad_phi_mag": float(np.mean(self.grad_phi_mag)),
            "mean_grad_R_mag": float(np.mean(self.grad_R_mag)),
            "mean_I_mag": float(np.mean(self.I_mag)),

            # Optional reporting items retained as diagnostics only
            # F_flip is reported here but does not influence motion
            "mean_abs_C_rec": float(np.mean(np.abs(self.C_rec))),
            "mean_F_flip": float(np.mean(self.F_flip)),
            "mean_M_mag": float(np.mean(self.M_mag)),
            "mean_A_hist": float(np.mean(self.A_hist)),

            "boundary_occupancy_ratio": boundary_occupancy_ratio,
            "rho_boundary_enrichment": float(rho_boundary_enrichment),
            "bulk_oscillation_ratio": float(bulk_oscillation_ratio),
            "rho_boundary_share": rho_boundary_share,

            "mean_flux_proj": float(np.mean(np.abs(self.flux_proj))),
            "mean_decoherence_term": float(np.mean(np.abs(self.decoherence_term))),

            "pos_fraction": float(np.mean(self.u_curr > 0.0)),
            "neg_fraction": float(np.mean(self.u_curr < 0.0)),
            "u_min": float(np.min(self.u_curr)),
            "u_max": float(np.max(self.u_curr)),
        }

    def _capture_summary(self, step_index: int) -> None:
        summary = self.compute_summary(step_index)
        self.summaries.append(summary)

        if step_index % self.cfg.summary_every == 0 or step_index == 0:
            print(
                f"step {summary['step']:4d} | "
                f"<|u|> {summary['mean_abs_u']:.3e} | "
                f"<rho> {summary['mean_rho']:.3e} | "
                f"<phi> {summary['mean_phi']:.3e} | "
                f"<Gamma> {summary['mean_Gamma']:.3e} | "
                f"<R> {summary['mean_R']:.3e} | "
                f"<Gamma_D> {summary['mean_Gamma_D']:.3e} | "
                f"<|grad_phi|> {summary['mean_grad_phi_mag']:.3e} | "
                f"<|grad_R|> {summary['mean_grad_R_mag']:.3e} | "
                f"<|I|> {summary['mean_I_mag']:.3e} | "
                f"<|C_rec|> {summary['mean_abs_C_rec']:.3e} | "
                f"<F_flip> {summary['mean_F_flip']:.3e} | "
                f"<M_mag> {summary['mean_M_mag']:.3e} | "
                f"<A_hist> {summary['mean_A_hist']:.3e} | "
                f"r_boundary {summary['boundary_occupancy_ratio']:.3e} | "
                f"E_rho_boundary {summary['rho_boundary_enrichment']:.3e} | "
                f"O_bulk {summary['bulk_oscillation_ratio']:.3e} | "
                f"S_rho_boundary {summary['rho_boundary_share']:.3e} | "
                f"<|flux|> {summary['mean_flux_proj']:.3e} | "
                f"<|decohere|> {summary['mean_decoherence_term']:.3e} | "
                f"+frac {summary['pos_fraction']:.3e} | "
                f"-frac {summary['neg_fraction']:.3e} | "
                f"u_min {summary['u_min']:.3e} | "
                f"u_max {summary['u_max']:.3e}"
            )

    def compute_final_summary(self) -> Dict[str, Any]:
        """
        Final summary for end-of-run reporting.

        Uses the last captured summary if available, then appends the
        run-defining configuration settings needed to identify the regime.
        """
        if self.summaries:
            summary = dict(self.summaries[-1])
        else:
            summary = self.compute_summary(step_index=-1)

        summary.update({
            "grid_h": int(self.cfg.grid_h),
            "grid_w": int(self.cfg.grid_w),
            "steps": int(self.cfg.steps),

            "w": int(self.cfg.w),
            "alpha": float(self.cfg.alpha),
            "rho_c": float(self.cfg.rho_c),

            "u0": float(self.cfg.u0),
            "seed_sign": int(self.cfg.seed_sign),

            "lambda_delta": float(self.cfg.lambda_delta),
            "lambda_flux": float(self.cfg.lambda_flux),
            "lambda_decoherence": float(self.cfg.lambda_decoherence),
            "operator_mode": str(self.cfg.operator_mode),
        })

        return summary

    def _capture_frames(self, step_index: int) -> None:
        if step_index % self.cfg.capture_every != 0:
            return

        if self.cfg.store_u_frames:
            self.u_frames.append(self.u_curr.copy())

        if self.cfg.store_R_frames:
            self.R_frames.append(self.R_proxy.copy())

    # =========================================================
    # Optional validation helpers
    # =========================================================
    def validate_basic_state(self) -> Dict[str, Any]:
        return {
            "has_nan_u": bool(np.isnan(self.u_curr).any()),
            "has_inf_u": bool(np.isinf(self.u_curr).any()),
            "has_nan_R": bool(np.isnan(self.R_proxy).any()),
            "has_inf_R": bool(np.isinf(self.R_proxy).any()),
            "max_abs_u": float(np.max(np.abs(self.u_curr))),
            "max_R_proxy": float(np.max(self.R_proxy)),
            "min_R_proxy": float(np.min(self.R_proxy)),
        }

    # =========================================================
    # Visualization / MP4 export
    # =========================================================
    def export_u_mp4(self, output_file: str | None = None) -> str:
        """
        Export the captured u-field frames as an MP4 and attempt download in Colab.
        """
        if not self.u_frames:
            raise ValueError("No u-field frames are stored. Set store_u_frames=True and rerun.")

        output_file = output_file or self.cfg.output_mp4

        # Display-only amplification:
        # use one global symmetric percentile scale across all stored frames.
        # This does not modify simulation values; it only changes how colors
        # are mapped in the exported movie.
        all_abs = np.concatenate([np.abs(frame).ravel() for frame in self.u_frames])
        vmax = float(np.percentile(all_abs, self.cfg.export_display_percentile))
        vmax = max(vmax, 1e-12)

        fig, ax = plt.subplots(figsize=(7, 7))
        im = ax.imshow(self.u_frames[0], cmap=self.cfg.cmap, vmin=-vmax, vmax=vmax)

        # Initial title for frame 0
        frame0 = self.u_frames[0]
        u_max0 = float(np.max(frame0))
        u_min0 = float(np.min(frame0))
        pos_fraction0 = float(np.count_nonzero(frame0 > 0) / frame0.size)
        neg_fraction0 = float(np.count_nonzero(frame0 < 0) / frame0.size)
        seed_label = "+" if self.cfg.seed_sign > 0 else "-"

        ax.set_title(
            f"{self.cfg.plot_title} | Frame: 0 | Seed: {seed_label}\n"
            f"u_max/min: {u_max0:.3e}, {u_min0:.3e} | "
            f"POS/NEG: {pos_fraction0:.3e}, {neg_fraction0:.3e}"
        )

        if self.cfg.show_axis_ticks:
            # Display ticks as cell-count positions rather than raw indices.
            x_positions = np.linspace(0, self.wd - 1, 5)
            x_labels = [str(int(round(v))) for v in np.linspace(0, self.wd, 5)]

            ax.set_xticks(x_positions)
            ax.set_xticklabels(x_labels)
            ax.tick_params(axis="x", bottom=True, top=False, labelbottom=True)

            # Keep y-axis hidden
            ax.set_yticks([])
        else:
            ax.set_xticks([])
            ax.set_yticks([])

        def update(i: int):
            frame = self.u_frames[i]
            im.set_data(frame)

            u_max = float(np.max(frame))
            u_min = float(np.min(frame))
            pos_fraction = float(np.count_nonzero(frame > 0) / frame.size)
            neg_fraction = float(np.count_nonzero(frame < 0) / frame.size)
            seed_label = "+" if self.cfg.seed_sign > 0 else "-"

            ax.set_title(
                f"{self.cfg.plot_title} | Frame: {i} | Seed: {seed_label}\n"
                f"u_max/min: {u_max:.3e}, {u_min:.3e} | "
                f"POS/NEG: {pos_fraction:.3e}, {neg_fraction:.3e}"
            )

            return [im]

        ani = animation.FuncAnimation(
            fig,
            update,
            frames=len(self.u_frames),
            interval=self.cfg.interval_ms,
            blit=False,
        )

        plt.close(fig)

        ani.save(output_file, writer="ffmpeg", fps=self.cfg.fps)
        print(f"Saved MP4: {output_file}")

        try:
            from google.colab import files  # type: ignore
            files.download(output_file)
        except Exception:
            print(f"Download not triggered automatically. File is saved as: {output_file}")

        return output_file



## Run simulation

In [ ]:
cfg = Config(
    grid_h=300,    # (20, 60, 100, 300, largest tested 500)
    grid_w=300,    # (20, 60, 100, 300, largest tested 500)
    steps=1000,    # (1000)
    w=6,
    alpha=10.0,
    rho_c=0.5,
    u0=1e-3,
    lambda_delta=1.00,        # (1.00)
    lambda_flux=0.50,         # (0.50)
    lambda_decoherence=0.25,  # (0.25)
    seed_sign=1,  # (1 or -1)
    operator_mode="matched",  # ("matched" or "naive")
    summary_every=10,
    capture_every=1,
    fps=12,
    interval_ms=80,
    output_mp4="2D_u_field.mp4",
)

sim = MinimalSim(cfg)
results = sim.run()

print("\nFinal validation:")
print(sim.validate_basic_state())

print("\nFinal summary:")
print(sim.compute_final_summary())


## Export MP4 (optional)

In [ ]:
sim.export_u_mp4()
